In [8]:
import arcpy
from arcpy.sa import ExtractByMask
from arcpy.sa import Slope

# check out Spatial Analyst license
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

In [9]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input raster "GBC15a_bathy_entireEastCarib_g4326"
unclip_gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_g4326"

# Input boundary polygon to crop raster to
boundary = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb\boundary_thesis_p26920"


# Set output file paths for reprojected raster, cropped raster, and slope raster
reproj_unclip_gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_p26920_v6642"
gbc_bathy = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_bathy_p26920_v6642"
gbc_slope = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_slope_p26920_v6642"


In [10]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # no compression on output rasters

In [12]:
# ── 1. PROJECT FULL GBC RASTER TO NAD83 UTM ZONE 20N, VIVD09 HEIGHT ──────────────────────────────────

# Project Raster (Data Management Tools):
arcpy.management.ProjectRaster(
    in_raster=unclip_gbc_bathy,
    out_raster=reproj_unclip_gbc_bathy,
    out_coor_system=arcpy.SpatialReference(26920, 6642),    # NAD 1983 UTM zone 20N (EPSG:26920), VIVD09_height (EPSG:6642)
    resampling_type="BILINEAR",
    geographic_transform="WGS_1984_(ITRF00)_To_NAD_1983",
    # leave NoData as default ("") to preserve NoData (land) from input raster
)

# Build pyramids with bilinear resampling, no compression
arcpy.management.BuildPyramids(
    reproj_unclip_gbc_bathy,
    resample_technique="BILINEAR",
    compression_type="NONE",
)

# Calculate raster statistics, X and Y skip factor 1
arcpy.management.CalculateStatistics(
    reproj_unclip_gbc_bathy,
    x_skip_factor=1,
    y_skip_factor=1,
)

# Print min and max values to check NoData (land areas) correctly preserved
reproj_min = arcpy.management.GetRasterProperties(reproj_unclip_gbc_bathy, "MINIMUM").getOutput(0)
reproj_max = arcpy.management.GetRasterProperties(reproj_unclip_gbc_bathy, "MAXIMUM").getOutput(0)
print(f"Reprojected raster min: {reproj_min}, max: {reproj_max}")


Reprojected raster min: -8619, max: 0


In [13]:
# ── 2. CROP RASTER TO MISSION BOUNDARY POLYGON ───────────────────────────────────────────────────────

# Set snap raster to reprojected raster so cropped output aligns to same grid (*since in 'environments' window of tool GUI, must set before running tool in script)
arcpy.env.snapRaster = reproj_unclip_gbc_bathy

# Extract by Mask (Spatial Analyst Tools): 
# input raster: reproj_unclip_gbc_bathy, input raster or feature mask data: boundary, extraction area: inside
out_bathy = ExtractByMask(reproj_unclip_gbc_bathy, boundary, "INSIDE")
# output raster: gbc_bathy
out_bathy.save(gbc_bathy)

# Print confirmation if raster was successfully created
if arcpy.Exists(gbc_bathy):
    print(f"Cropped bathy raster successfully created: {gbc_bathy}")

Cropped bathy raster successfully created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_bathy_p26920_v6642


In [14]:
# ── 3. CREATE GBC SLOPE RASTER FROM BATHY RASTER ─────────────────────────────────────────────────────

# keep snap raster aligned to reprojected raster grid so output aligns to same grid
arcpy.env.snapRaster = reproj_unclip_gbc_bathy

# Slope (Spatial Analyst Tools): 
# input raster: gbc_bathy, output measurement: degree, method: planar
out_slope = Slope(gbc_bathy, "DEGREE", method="PLANAR")
# output raster: gbc_slope
out_slope.save(gbc_slope)

# Print confirmation if raster was successfully created and min and max values in output raster
if arcpy.Exists(gbc_slope):
    print(f"Slope raster successfully created: {gbc_slope}")
    slope_min = arcpy.management.GetRasterProperties(gbc_slope, "MINIMUM").getOutput(0)
    slope_max = arcpy.management.GetRasterProperties(gbc_slope, "MAXIMUM").getOutput(0)
    print(f"Slope raster min: {slope_min}, max: {slope_max}")

Slope raster successfully created: C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb\GBC15a_slope_p26920_v6642
Slope raster min: 0, max: 79.8360061645508


In [15]:
# release Spatial Analyst license
arcpy.CheckInExtension("Spatial")

'CheckedOut'